# Logistic Regression: Student Performance Factors

## Step 1: Import Libraries

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

## Step 2: Load and Explore Data

In [ ]:
df = pd.read_csv(r"C:\Users\24108218\Documents\Downloads\s\StudentPerformanceFactors.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Step 3: Data Cleaning

In [ ]:
print(df.isnull().sum())
df = df.drop_duplicates()
df = df.dropna()
print(df.shape)

## Step 4: Create Binary Target (Pass = score >= 70)

In [ ]:
df['Result'] = (df['Exam_Score'] >= 70).astype(int)   # 1 = Pass, 0 = Fail
print(df['Result'].value_counts())
sns.countplot(x='Result', data=df)
plt.title('Class distribution (0 = Fail, 1 = Pass)')
plt.show()

## Step 4b: Define Features (X) and Target (y)

In [ ]:
num_cols = ['Hours_Studied', 'Attendance', 'Previous_Scores',
            'Sleep_Hours', 'Tutoring_Sessions', 'Physical_Activity']
cat_cols = ['Parental_Involvement', 'Motivation_Level']

X = df[num_cols + cat_cols]   # Exam_Score / Result excluded to avoid leakage
y = df['Result']

print(X.shape, y.shape)

## Step 5: Split, Preprocess and Train

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

model = Pipeline([
    ('prep', preprocess),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

model.fit(X_train, y_train)

## Step 6: Model Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Fail','Pass']))

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=['Fail','Pass']).plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## Step 7: Test One Prediction and Save the Model

In [ ]:
sample = pd.DataFrame([{
    'Hours_Studied': 25, 'Attendance': 90, 'Previous_Scores': 80,
    'Sleep_Hours': 7, 'Tutoring_Sessions': 2, 'Physical_Activity': 3,
    'Parental_Involvement': 'High', 'Motivation_Level': 'High'}])
print("Prediction:", "Pass" if model.predict(sample)[0] == 1 else "Fail")
print("Probability of Pass:", model.predict_proba(sample)[0][1])

joblib.dump(model, 'student_model.pkl')
print("Model saved: student_model.pkl")